In [22]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
from abc import ABC, abstractmethod
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Intentar importar plotly, si falla dar mensaje
try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠️ Plotly no está instalado. La vista 3D interactiva no funcionará.")
    print("   Instálalo con: !pip install plotly")

# ============================================================
# 1. CLASE ABSTRACTA
# ============================================================
class SistemaFisico(ABC):
    @abstractmethod
    def derivada(self, t, estado):
        pass

    @abstractmethod
    def simular(self, t_span, estado_inicial, t_eval=None):
        pass

# ============================================================
# 2. TERRENO (con varias superficies predefinidas)
# ============================================================
class Terreno:
    def __init__(self, funcion_altura):
        self.altura = funcion_altura

    def gradiente(self, x, y, h=1e-5):
        dzdx = (self.altura(x+h, y) - self.altura(x-h, y)) / (2*h)
        dzdy = (self.altura(x, y+h) - self.altura(x, y-h)) / (2*h)
        return dzdx, dzdy

    def energia_potencial(self, x, y, masa, g):
        return masa * g * self.altura(x, y)

# Diccionario de terrenos disponibles
TERRENOS = {
    "Montaña clásica": lambda x, y: (
        np.sin(0.8*x)*np.cos(0.8*y) +
        0.5*np.sin(1.5*x)*np.cos(1.2*y) +
        0.3*np.exp(-(x**2+y**2)/4)
    ),
    "Plano inclinado + montículos": lambda x, y: (
        -0.6*x +
        0.8*np.exp(-((x-1)**2+(y-1)**2)/1.2) +
        0.6*np.exp(-((x+0.5)**2+(y+1)**2)/1.0) -
        0.5*np.exp(-((x+1)**2+(y-1)**2)/1.5)
    ),
    "Dos picos y valle central": lambda x, y: (
        1.2*np.exp(-((x+1.5)**2+(y+1.5)**2)/1.5) +
        1.2*np.exp(-((x-1.5)**2+(y-1.5)**2)/1.5) -
        0.8*np.exp(-(x**2+y**2)/2.0) +
        0.1*np.sin(1.2*x)*np.cos(1.2*y)
    ),
    "Pozo de potencial atractivo": lambda x, y: (
        -1.5*np.exp(-(x**2+y**2)/1.2) +
        0.2*np.sin(1.5*x)*np.cos(1.5*y)
    ),
    "Silla de montar": lambda x, y: (
        0.8*(x**2 - y**2)*np.exp(-(x**2+y**2)/6) -
        0.5*np.exp(-((x-1)**2+y**2)/2)
    ),
    "Montaña rugosa (altas frecuencias)": lambda x, y: (
        np.sin(0.8*x)*np.cos(0.8*y) +
        0.5*np.sin(1.5*x)*np.cos(1.2*y) +
        0.3*np.sin(2.2*x)*np.cos(2.0*y) +
        0.2*np.sin(3.0*x)*np.cos(2.5*y) +
        0.3*np.exp(-(x**2+y**2)/4)
    )
}

# ============================================================
# 3. PARTÍCULA (con evento de parada por velocidad baja)
# ============================================================
class PararPorVelocidad:
    """Clase evento para solve_ivp: detiene cuando la rapidez cae por debajo del umbral."""
    def __init__(self, v_umbral):
        self.v_umbral = v_umbral
        self.terminal = True
        self.direction = -1   # cruce de positivo a negativo

    def __call__(self, t, estado):
        vx, vy = estado[2], estado[3]
        return np.hypot(vx, vy) - self.v_umbral

class Particula(SistemaFisico):
    def __init__(self, terreno, masa=1.0, rozamiento=0.1, gravedad=9.8, v_umbral=1e-2):
        self.terreno = terreno
        self.masa = masa
        self.rozamiento = rozamiento
        self.gravedad = gravedad
        self.v_umbral = v_umbral

    def derivada(self, t, estado):
        x, y, vx, vy = estado
        dzdx, dzdy = self.terreno.gradiente(x, y)
        ax = -self.gravedad * dzdx - self.rozamiento * vx
        ay = -self.gravedad * dzdy - self.rozamiento * vy
        return [vx, vy, ax, ay]

    def simular(self, t_span, estado_inicial, t_eval=None):
        evento = PararPorVelocidad(self.v_umbral)
        sol = solve_ivp(
            self.derivada, t_span, estado_inicial,
            t_eval=t_eval, method='RK45',
            events=evento,
            rtol=1e-6, atol=1e-8
        )
        return sol.t, sol.y

# ============================================================
# 4. SIMULADOR INTERACTIVO
# ============================================================
class SimuladorInteractivo:
    def __init__(self):
        self.terreno = Terreno(TERRENOS["Montaña clásica"])
        self.tipo_terreno_actual = "Montaña clásica"

        self.masa = 1.0
        self.rozamiento = 0.1
        self.gravedad = 9.8
        self.t_max = 200.0

        self.posicion = np.array([-2.0, 1.8])
        self.velocidad = np.array([-2.0, -4.0])

        self.trayectoria = None
        self.tiempos = None
        self.energias = None
        self.particula = None

        self.crear_interfaz()

    def crear_interfaz(self):
        # Selector de terreno
        self.terreno_selector = widgets.Dropdown(
            options=list(TERRENOS.keys()),
            value=self.tipo_terreno_actual,
            description='Terreno:',
            style={'description_width': 'initial'}
        )

        # Posición inicial
        self.slider_x = widgets.FloatSlider(value=-2.0, min=-3.0, max=3.0, step=0.1, description='Pos X:', style={'description_width': 'initial'})
        self.slider_y = widgets.FloatSlider(value=1.8, min=-2.8, max=2.8, step=0.1, description='Pos Y:', style={'description_width': 'initial'})

        # Velocidad inicial
        self.slider_vx = widgets.FloatSlider(value=-2.0, min=-5.0, max=5.0, step=0.1, description='Vel X:', style={'description_width': 'initial'})
        self.slider_vy = widgets.FloatSlider(value=-4.0, min=-5.0, max=5.0, step=0.1, description='Vel Y:', style={'description_width': 'initial'})

        # Parámetros físicos
        self.slider_roz = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description='Rozamiento μ:', style={'description_width': 'initial'})
        self.slider_grav = widgets.FloatSlider(value=9.8, min=2.0, max=20.0, step=0.5, description='Gravedad g:', style={'description_width': 'initial'})
        self.slider_masa = widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1, description='Masa (kg):', style={'description_width': 'initial'})

        # Controles 3D
        self.slider_elev = widgets.IntSlider(value=30, min=0, max=90, step=5, description='Elevación °:', style={'description_width': 'initial'})
        self.slider_azim = widgets.IntSlider(value=225, min=0, max=360, step=5, description='Azimut °:', style={'description_width': 'initial'})
        self.btn_actualizar_3d = widgets.Button(description="🔄 Actualizar vista 3D", button_style='warning')

        # Botones principales
        self.btn_simular = widgets.Button(description="▶️ Simular", button_style='success')
        self.btn_analisis = widgets.Button(description="📊 Análisis avanzado (curve_fit)", button_style='info')

        # Consulta de energía
        self.consulta_x = widgets.FloatText(value=0.0, description='x:', step=0.1)
        self.consulta_y = widgets.FloatText(value=0.0, description='y:', step=0.1)
        self.btn_consulta = widgets.Button(description="🔍 Energía potencial en (x,y)")

        # Áreas de salida
        self.out_2d = widgets.Output()
        self.out_energia = widgets.Output()
        self.out_3d = widgets.Output()
        self.out_consulta = widgets.Output()
        self.out_analisis = widgets.Output()

        # Layout
        controles_pos = widgets.VBox([widgets.HTML("<b>Posición inicial</b>"), self.slider_x, self.slider_y])
        controles_vel = widgets.VBox([widgets.HTML("<b>Velocidad inicial</b>"), self.slider_vx, self.slider_vy])
        controles_fis = widgets.HBox([self.slider_roz, self.slider_grav, self.slider_masa])
        controles_3d = widgets.VBox([
            widgets.HTML("<b>Vista 3D (Plotly)</b>"),
            self.slider_elev, self.slider_azim, self.btn_actualizar_3d
        ])
        panel_consulta = widgets.HBox([self.consulta_x, self.consulta_y, self.btn_consulta])

        self.ui = widgets.VBox([
            self.terreno_selector,
            widgets.HBox([controles_pos, controles_vel]),
            controles_fis,
            widgets.HBox([self.btn_simular, self.btn_analisis]),
            controles_3d,
            panel_consulta,
            self.out_consulta,
            widgets.HTML("<hr><b>Mapa 2D + Trayectoria</b>"),
            self.out_2d,
            widgets.HTML("<b>Energías vs Tiempo</b>"),
            self.out_energia,
            widgets.HTML("<b>Vista 3D interactiva (Plotly)</b>"),
            self.out_3d,
            widgets.HTML("<hr>"),
            self.out_analisis
        ])

        # Conectar eventos
        self.btn_simular.on_click(self.ejecutar_simulacion)
        self.btn_analisis.on_click(self.ejecutar_analisis)
        self.btn_consulta.on_click(self.mostrar_energia_punto)
        self.btn_actualizar_3d.on_click(self.actualizar_vista_3d)

        display(self.ui)

    def ejecutar_simulacion(self, _):
        # Cambiar terreno si es necesario
        nuevo = self.terreno_selector.value
        if nuevo != self.tipo_terreno_actual:
            self.tipo_terreno_actual = nuevo
            self.terreno = Terreno(TERRENOS[nuevo])

        # Leer valores de los sliders
        self.posicion = np.array([self.slider_x.value, self.slider_y.value])
        self.velocidad = np.array([self.slider_vx.value, self.slider_vy.value])
        self.masa = self.slider_masa.value
        self.rozamiento = self.slider_roz.value
        self.gravedad = self.slider_grav.value

        # Crear partícula
        self.particula = Particula(self.terreno, self.masa, self.rozamiento, self.gravedad, v_umbral=0.005)
        estado0 = [self.posicion[0], self.posicion[1], self.velocidad[0], self.velocidad[1]]

        # Simular
        self.tiempos, y_data = self.particula.simular((0, self.t_max), estado0, t_eval=None)

        if y_data.size == 0:
            print("❌ La simulación no produjo datos. Verifique los parámetros.")
            return

        estados = y_data.T  # (N, 4)
        self.trayectoria = estados[:, :2]

        # Calcular energías
        Ec = 0.5 * self.masa * (estados[:, 2]**2 + estados[:, 3]**2)
        Ep = np.array([self.terreno.energia_potencial(x, y, self.masa, self.gravedad) for x, y in self.trayectoria])
        self.energias = np.column_stack([Ec, Ep, Ec + Ep])

        # Graficar
        self._graficar_2d()
        self._graficar_energia()
        if PLOTLY_AVAILABLE:
            self._graficar_3d_plotly()
        else:
            with self.out_3d:
                clear_output(wait=True)
                print("Plotly no está instalado. Vista 3D no disponible.")

        # Información de parada
        v_final = np.hypot(estados[-1, 2], estados[-1, 3])
        print(f"\n⏱️ Simulación terminada en t = {self.tiempos[-1]:.2f} s")
        print(f"   Velocidad final: {v_final:.4f} m/s")
        if self.tiempos[-1] >= self.t_max - 0.5:
            print("⚠️ Se alcanzó t_max sin detenerse. Aumente t_max o reduzca v_umbral.")
        else:
            print("✅ La canica se detuvo (velocidad < umbral).")

    def _graficar_2d(self):
        xg = np.linspace(-3, 3, 80)
        yg = np.linspace(-3, 3, 80)
        X, Y = np.meshgrid(xg, yg)
        Z = self.terreno.altura(X, Y)

        with self.out_2d:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 6))
            cont = ax.contourf(X, Y, Z, levels=30, cmap='terrain', alpha=0.85)
            ax.contour(X, Y, Z, levels=10, colors='black', linewidths=0.3, alpha=0.5)
            ax.plot(self.trayectoria[:, 0], self.trayectoria[:, 1], 'r-', lw=2, label='Trayectoria')
            ax.scatter(*self.trayectoria[0], c='lime', s=100, zorder=5, label='Inicio', edgecolors='k')
            ax.scatter(*self.trayectoria[-1], c='blue', s=100, zorder=5, label='Fin', edgecolors='k')
            ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect('equal')
            ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
            ax.set_title(f'Trayectoria — {self.tipo_terreno_actual}')
            ax.legend()
            plt.colorbar(cont, ax=ax, label='Altura (m)')
            plt.tight_layout()
            display(fig)
            plt.close(fig)

    def _graficar_energia(self):
        with self.out_energia:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(self.tiempos, self.energias[:, 0], label='Cinética', color='orangered')
            ax.plot(self.tiempos, self.energias[:, 1], label='Potencial', color='steelblue')
            ax.plot(self.tiempos, self.energias[:, 2], 'k--', lw=1.5, label='Total')
            ax.set_xlabel('Tiempo (s)'); ax.set_ylabel('Energía (J)')
            ax.set_title('Evolución de energías')
            ax.legend(); ax.grid(True, alpha=0.4)
            plt.tight_layout()
            display(fig)
            plt.close(fig)

    def _graficar_3d_plotly(self):
        xg = np.linspace(-3, 3, 60)
        yg = np.linspace(-3, 3, 60)
        X, Y = np.meshgrid(xg, yg)
        Z = self.terreno.altura(X, Y)
        z_tray = np.array([self.terreno.altura(x, y) for x, y in self.trayectoria])

        elev = self.slider_elev.value
        azim = self.slider_azim.value
        # Convertir ángulos a vector eye para Plotly
        elev_rad = np.radians(elev)
        azim_rad = np.radians(azim)
        r = 2.0
        eye_x = r * np.cos(elev_rad) * np.cos(azim_rad)
        eye_y = r * np.cos(elev_rad) * np.sin(azim_rad)
        eye_z = r * np.sin(elev_rad)

        fig = go.Figure()
        fig.add_trace(go.Surface(
            x=X, y=Y, z=Z,
            colorscale='earth', opacity=0.80,
            showscale=True, colorbar=dict(title='Altura (m)', len=0.6),
            name='Terreno'
        ))
        fig.add_trace(go.Scatter3d(
            x=self.trayectoria[:, 0], y=self.trayectoria[:, 1], z=z_tray,
            mode='lines', line=dict(color='red', width=4),
            name='Trayectoria'
        ))
        fig.add_trace(go.Scatter3d(
            x=[self.trayectoria[0, 0]], y=[self.trayectoria[0, 1]], z=[z_tray[0]],
            mode='markers', marker=dict(color='lime', size=8, symbol='circle', line=dict(color='black', width=1)),
            name='Inicio'
        ))
        fig.add_trace(go.Scatter3d(
            x=[self.trayectoria[-1, 0]], y=[self.trayectoria[-1, 1]], z=[z_tray[-1]],
            mode='markers', marker=dict(color='royalblue', size=8, symbol='square', line=dict(color='black', width=1)),
            name='Fin'
        ))
        fig.update_layout(
            title=dict(text=f'Trayectoria 3D — {self.tipo_terreno_actual}', x=0.5),
            scene=dict(
                xaxis=dict(range=[-3, 3], title='X (m)'),
                yaxis=dict(range=[-3, 3], title='Y (m)'),
                zaxis=dict(title='Altura (m)'),
                camera=dict(eye=dict(x=eye_x, y=eye_y, z=eye_z))
            ),
            width=750, height=560,
            margin=dict(l=10, r=10, t=50, b=10),
            legend=dict(x=0.01, y=0.99)
        )
        with self.out_3d:
            clear_output(wait=True)
            display(fig)

    def actualizar_vista_3d(self, _):
        if self.trayectoria is None:
            with self.out_3d:
                clear_output(wait=True)
                print("⚠️ Primero ejecuta una simulación.")
            return
        if PLOTLY_AVAILABLE:
            self._graficar_3d_plotly()
        else:
            with self.out_3d:
                clear_output(wait=True)
                print("Plotly no está instalado. No se puede actualizar la vista.")

    def mostrar_energia_punto(self, _):
        with self.out_consulta:
            clear_output(wait=True)
            if self.particula is None:
                print("⚠️ Primero ejecuta una simulación.")
                return
            x = self.consulta_x.value
            y = self.consulta_y.value
            Ep = self.terreno.energia_potencial(x, y, self.masa, self.gravedad)
            altura = self.terreno.altura(x, y)
            print(f"📍 Punto ({x:.2f}, {y:.2f})")
            print(f"   Altura:            {altura:.4f} m")
            print(f"   Energía potencial: {Ep:.4f} J  (masa={self.masa:.2f} kg, g={self.gravedad:.1f} m/s²)")

    def ejecutar_analisis(self, _):
        with self.out_analisis:
            clear_output(wait=True)
            print("=" * 60)
            print("ANÁLISIS: Ajuste de parámetros con scipy.optimize.curve_fit")
            print("=" * 60)

            terreno_actual = self.terreno
            roz_real = 0.12
            g_real = 9.8
            masa = 1.0
            estado0 = [1.0, 1.5, 0.0, 0.0]
            t_max = 10.0
            t_med = np.linspace(0, t_max, 150)

            particula_real = Particula(terreno_actual, masa, roz_real, g_real)
            _, y_real = particula_real.simular((0, t_max), estado0, t_eval=t_med)
            ruido = 0.03
            x_med = y_real[0] + np.random.normal(0, ruido, size=t_med.shape)
            y_med = y_real[1] + np.random.normal(0, ruido, size=t_med.shape)

            def modelo_ajuste(t, roz, g):
                part_temp = Particula(terreno_actual, masa, roz, g)
                _, y_temp = part_temp.simular((0, t_max), estado0, t_eval=t)
                return np.concatenate([y_temp[0], y_temp[1]])

            ydata = np.concatenate([x_med, y_med])
            try:
                popt, pcov = curve_fit(modelo_ajuste, t_med, ydata,
                                       p0=[0.2, 9.5], maxfev=6000,
                                       bounds=([0.0, 2.0], [2.0, 20.0]))
                perr = np.sqrt(np.diag(pcov))
                roz_aj, g_aj = popt

                modelo_aj = Particula(terreno_actual, masa, roz_aj, g_aj)
                _, y_aj = modelo_aj.simular((0, t_max), estado0, t_eval=t_med)

                mse = np.mean((x_med - y_aj[0])**2 + (y_med - y_aj[1])**2)
                mae = np.mean(np.abs(x_med - y_aj[0]) + np.abs(y_med - y_aj[1]))

                print(f"\nRozamiento real : {roz_real:.3f}  →  ajustado : {roz_aj:.4f} ± {perr[0]:.4f}")
                print(f"Gravedad real   : {g_real:.1f}    →  ajustada : {g_aj:.4f} ± {perr[1]:.4f}")
                print(f"\nError cuadrático medio (MSE): {mse:.6f}")
                print(f"Error absoluto medio   (MAE): {mae:.6f}")
                print("\n✅ Análisis completado con scipy.optimize.curve_fit")
            except Exception as e:
                print(f"❌ Error en el ajuste: {e}")

# ============================================================
# 5. EJECUCIÓN PRINCIPAL
# ============================================================
print("🎯 SIMULADOR INTERACTIVO DE CANICA")
print("   Terreno parametrizable | Parada automática | 3D interactivo (Plotly)")
print("-" * 60)
sim = SimuladorInteractivo()

🎯 SIMULADOR INTERACTIVO DE CANICA
   Terreno parametrizable | Parada automática | 3D interactivo (Plotly)
------------------------------------------------------------



⏱️ Simulación terminada en t = 75.00 s
   Velocidad final: 0.0050 m/s
✅ La canica se detuvo (velocidad < umbral).

⏱️ Simulación terminada en t = 48.21 s
   Velocidad final: 0.0050 m/s
✅ La canica se detuvo (velocidad < umbral).

⏱️ Simulación terminada en t = 77.55 s
   Velocidad final: 0.0050 m/s
✅ La canica se detuvo (velocidad < umbral).
